# 10 — Page-as-Image Retrieval with ColPali — Beyond Text-Based Document Search

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/10_page_as_image_retrieval_with_colpali.ipynb)

> **Orbit 5 (Multimodal)** · **Difficulty**: Intermediate · **Reading time**: ~30 min

Document retrieval has historically been a text-only affair: scan a page with OCR,
chunk the extracted text, embed each chunk, and search with cosine similarity.
That pipeline works well for text-heavy pages, but it **discards layout, tables,
figures, logos, and colour cues** — exactly the signals that make a human glance
at a page and instantly know what it contains. ColPali flips the script: it embeds
the raw page image and lets the model keep every visual detail intact.

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

1. **Explain** why traditional OCR → text-embed → search pipelines lose information and when that matters.
2. **Describe** the ColPali architecture — PaliGemma vision encoder plus ColBERT-style late interaction.
3. **Encode** document page images into multi-vector patch embeddings with the `colpali-engine` library.
4. **Compute** MaxSim scores between query tokens and page patches, and interpret the resulting heatmaps.
5. **Build** a FAISS-backed retrieval index over page embeddings and compare ColPali retrieval against a text-only baseline.

## Section 1 — The Paradigm Shift: From OCR Pipelines to Vision-First Retrieval

### The Traditional Pipeline

Most document-search systems follow a four-step recipe:

1. **Rasterise & OCR** — render the PDF page and run an OCR engine.
2. **Chunk** — split the extracted text into overlapping passages.
3. **Embed** — encode each chunk with a text embedding model.
4. **Search** — embed the query and retrieve nearest chunks by cosine similarity.

This works well for narrative text, but it has a critical blind spot:
**any information not captured by the character stream is invisible**.
Tables become garbled rows. Figures are reduced to alt-text or nothing.
Spatial relationships — obvious to a human eye — are lost entirely.

### What ColPali Changes

ColPali sidesteps OCR altogether. It takes the rendered page image and
produces **patch-level embeddings** — one vector for every spatial region.
A query is encoded into its own set of vectors. Matching happens through
**late interaction** (MaxSim from ColBERT): every query token finds the
page patch it most resembles, and the scores are summed.

The diagram below contrasts the two pipelines.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Traditional pipeline ---
ax = axes[0]
ax.set_title("Traditional: OCR → Text Embed → Search", fontsize=12, fontweight="bold")
steps_trad = ["PDF Page", "OCR Engine", "Raw Text", "Chunking", "Text Embeddings", "Cosine Search"]
colors_trad = ["#cce5ff", "#ffe0b2", "#c8e6c9", "#ffe0b2", "#d1c4e9", "#f8bbd0"]
for i, (s, c) in enumerate(zip(steps_trad, colors_trad)):
    y = 1.0 - i * 0.16
    ax.add_patch(mpatches.FancyBboxPatch((0.15, y - 0.05), 0.7, 0.10,
                 boxstyle="round,pad=0.02", facecolor=c, edgecolor="grey"))
    ax.text(0.5, y, s, ha="center", va="center", fontsize=10)
    if i < len(steps_trad) - 1:
        ax.annotate("", xy=(0.5, y - 0.06), xytext=(0.5, y - 0.10),
                    arrowprops=dict(arrowstyle="->", color="grey"))
ax.set_xlim(0, 1); ax.set_ylim(0, 1.1); ax.axis("off")

# --- ColPali pipeline ---
ax = axes[1]
ax.set_title("ColPali: Page Image → Patch Embed → MaxSim", fontsize=12, fontweight="bold")
steps_col = ["PDF Page (image)", "Vision Encoder", "Patch Embeddings", "MaxSim Search"]
colors_col = ["#cce5ff", "#b2dfdb", "#d1c4e9", "#f8bbd0"]
for i, (s, c) in enumerate(zip(steps_col, colors_col)):
    y = 1.0 - i * 0.22
    ax.add_patch(mpatches.FancyBboxPatch((0.15, y - 0.07), 0.7, 0.12,
                 boxstyle="round,pad=0.02", facecolor=c, edgecolor="grey"))
    ax.text(0.5, y, s, ha="center", va="center", fontsize=10)
    if i < len(steps_col) - 1:
        ax.annotate("", xy=(0.5, y - 0.08), xytext=(0.5, y - 0.14),
                    arrowprops=dict(arrowstyle="->", color="grey"))
ax.set_xlim(0, 1); ax.set_ylim(0, 1.1); ax.axis("off")

plt.tight_layout()
plt.show()

## Environment Setup

We install the `colpali-engine` wrapper (which pulls in the right PaliGemma
checkpoints), FAISS for indexing, and `sentence-transformers` for the text baseline.

In [ ]:
!pip install -q colpali-engine transformers torch pillow bitsandbytes accelerate sentence-transformers faiss-cpu

In [ ]:
import torch, os, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## Section 2 — ColPali Architecture Deep Dive

ColPali combines two powerful ideas that were developed independently and merges
them into a single retrieval model:

| Component | Origin | Role in ColPali |
|---|---|---|
| **PaliGemma** vision encoder | Google (2024) | Converts a page image into a grid of patch embeddings — one vector per spatial region. |
| **Late interaction** (ColBERT) | Khattab & Zaharia (2020) | Scores query–document pairs by aligning individual query tokens with individual document vectors via MaxSim. |

### How It Works Step by Step

1. **Image patching** — The page image is resized and split into a regular grid of
   patches (typically 16×16 pixels each). Each patch is projected into the
   transformer embedding space.
2. **Vision transformer forward pass** — The patch sequence is processed by the
   PaliGemma encoder, producing contextualised patch embeddings that attend to
   each other through self-attention. A linear projection maps them to a shared
   query-document embedding dimension (128-d by default).
3. **Multi-vector representation** — Unlike single-vector models that collapse an
   entire document into one point, ColPali retains **all** patch vectors for every
   page. This preserves spatial information and lets different query tokens match
   different page regions.
4. **Query encoding** — The text query is tokenised and passed through a lightweight
   text tower that projects each token into the same 128-d space.
5. **MaxSim scoring** — For each query token, compute cosine similarity with
   every page patch, keep the maximum, and sum across all query tokens.

The key insight is that multi-vector late interaction lets ColPali match a query
about "Q3 revenue" to a specific table cell, even without OCR.

In [ ]:
from colpali_engine.models import ColPali, ColPaliProcessor

MODEL_NAME = "vidore/colpali-v1.3"

colpali = ColPali.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
colpali_processor = ColPaliProcessor.from_pretrained(MODEL_NAME)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters  : {sum(p.numel() for p in colpali.parameters()) / 1e6:.1f} M")
if device == "cuda":
    print(f"GPU memory  : {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## Section 3 — Encoding Document Pages

ColPali expects a PIL image as input. In a real system you would render each PDF
page with a library such as `pdf2image`, but here we will create **synthetic
document pages** so the notebook is self-contained and runs without external files.

Each synthetic page has a title, body text, and optionally a table or chart area.
This lets us control exactly what visual information is present and test whether
ColPali can retrieve the right page based on its content.

### Resolution and Resizing

ColPali internally resizes images to the resolution expected by PaliGemma
(typically 448 × 448 or 896 × 896 depending on the variant). Higher input
resolution does not help beyond the model's native size, but very low resolution
can hurt because fine text becomes unreadable. A good rule of thumb is to render
pages at **150–200 DPI** before passing them into the model.

In [ ]:
def create_document_page(title, body_lines, table_data=None,
                          width=800, height=1100):
    """Create a synthetic document page as a PIL Image."""
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        title_font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 28)
        body_font  = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 18)
    except OSError:
        title_font = ImageFont.load_default()
        body_font  = ImageFont.load_default()

    y = 40
    draw.text((40, y), title, fill="black", font=title_font)
    y += 50
    draw.line([(40, y), (width - 40, y)], fill="grey", width=2)
    y += 20

    for line in body_lines:
        draw.text((40, y), line, fill="#333333", font=body_font)
        y += 28

    if table_data:
        y += 15
        col_width = (width - 80) // len(table_data[0])
        for row_idx, row in enumerate(table_data):
            for col_idx, cell in enumerate(row):
                x0 = 40 + col_idx * col_width
                x1 = x0 + col_width
                y0, y1 = y, y + 30
                fill_color = "#e3f2fd" if row_idx == 0 else "white"
                draw.rectangle([x0, y0, x1, y1], outline="grey", fill=fill_color)
                draw.text((x0 + 8, y0 + 5), str(cell), fill="black", font=body_font)
            y += 30

    return img

print("create_document_page() defined.")

In [ ]:
# --- Create a small corpus of synthetic pages ---
pages = []
page_descriptions = []

# Page 0 — financial report with table
pages.append(create_document_page(
    title="Acme Corp — Q3 2024 Financial Report",
    body_lines=[
        "Revenue grew 12 % year-over-year driven by cloud services.",
        "Operating margin improved to 18.3 % from 15.7 % in Q2.",
        "",
        "Key financial metrics are summarised in the table below.",
    ],
    table_data=[
        ["Metric", "Q2 2024", "Q3 2024", "YoY Change"],
        ["Revenue ($M)", "340", "382", "+12 %"],
        ["Net Income ($M)", "53", "70", "+32 %"],
        ["EPS", "$1.06", "$1.40", "+32 %"],
    ],
))
page_descriptions.append("Financial report with revenue table")

# Page 1 — technical architecture
pages.append(create_document_page(
    title="System Architecture Overview",
    body_lines=[
        "The platform uses a microservices architecture deployed on Kubernetes.",
        "Services communicate via gRPC with Protocol Buffers for serialisation.",
        "A central API gateway handles authentication, rate limiting, and routing.",
        "Event-driven processing is handled by Apache Kafka with exactly-once semantics.",
        "Each service owns its own PostgreSQL database (database-per-service pattern).",
        "",
        "Monitoring is provided by Prometheus + Grafana dashboards.",
    ],
))
page_descriptions.append("Technical system architecture overview")

# Page 2 — HR policy
pages.append(create_document_page(
    title="Employee Benefits & Leave Policy",
    body_lines=[
        "Full-time employees are entitled to 20 days of paid time off per year.",
        "Sick leave is provided at 10 days per year and does not roll over.",
        "Parental leave: 16 weeks for primary caregivers, 6 weeks for secondary.",
        "Tuition reimbursement up to $5,000 per calendar year for approved courses.",
        "",
        "Health insurance covers medical, dental, and vision for employee + family.",
    ],
))
page_descriptions.append("HR benefits and leave policy")

# Page 3 — research paper abstract with table
pages.append(create_document_page(
    title="Efficient Attention Mechanisms: A Survey",
    body_lines=[
        "Transformer models have become the dominant architecture in NLP and vision.",
        "However, the quadratic cost of self-attention limits scalability.",
        "This survey reviews linear-attention variants, sparse patterns, and",
        "low-rank approximations that reduce complexity to O(n) or O(n log n).",
    ],
    table_data=[
        ["Method", "Complexity", "Accuracy"],
        ["Full Attention", "O(n^2)", "100 %"],
        ["Linformer", "O(n)", "97.3 %"],
        ["Performer", "O(n)", "96.1 %"],
    ],
))
page_descriptions.append("Research survey on efficient attention")

# Show thumbnails
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for i, (page, desc) in enumerate(zip(pages, page_descriptions)):
    axes[i].imshow(page)
    axes[i].set_title(f"Page {i}: {desc}", fontsize=9)
    axes[i].axis("off")
plt.tight_layout()
plt.show()
print(f"Created {len(pages)} synthetic document pages.")

In [ ]:
# --- Encode all pages with ColPali ---
batch_images = colpali_processor.process_images(pages).to(colpali.device)

with torch.no_grad():
    image_embeddings = colpali(**batch_images)  # list of tensors [num_patches, dim]

print(f"Number of pages encoded : {len(image_embeddings)}")
print(f"Patches per page        : {image_embeddings[0].shape[0]}")
print(f"Embedding dimension     : {image_embeddings[0].shape[1]}")
print(f"Dtype                   : {image_embeddings[0].dtype}")

## Section 4 — Late Interaction & MaxSim Scoring

Single-vector retrieval models compress an entire document into one point in
embedding space. That single point must somehow represent every topic, every
table cell, and every figure on the page — a tall order. Late interaction avoids
this bottleneck by keeping **multiple vectors** for both the query and the
document and deferring the comparison to scoring time.

### The MaxSim Operator

Given a query with $N_q$ token embeddings $\{q_1, \ldots, q_{N_q}\}$ and a
document page with $N_p$ patch embeddings $\{p_1, \ldots, p_{N_p}\}$, the
relevance score is:

$$
\text{score}(q, d) = \sum_{i=1}^{N_q} \max_{j=1}^{N_p}\ q_i^\top p_j
$$

For each query token we find the single most relevant page patch, then aggregate.
This lets a token like *"revenue"* match the table header cell while a token
like *"Q3"* matches the column header — fine-grained alignment that a single-
vector dot product cannot achieve.

### Why Late Interaction Is Powerful

* **No joint encoding** — query and document are encoded independently, so
  document embeddings can be precomputed and cached.
* **Fine-grained matching** — individual tokens interact with individual
  patches, preserving local information.
* **Efficient at scale** — the per-query cost is a matrix multiply over the
  cached patch matrix, not a full forward pass through a cross-encoder.

In [ ]:
# --- Encode a query and compute MaxSim manually ---
query_text = "What is the revenue for Q3 2024?"

batch_query = colpali_processor.process_queries([query_text]).to(colpali.device)
with torch.no_grad():
    query_emb = colpali(**batch_query)  # [1, num_tokens, dim]

print(f"Query tokens  : {query_emb.shape[1]}")
print(f"Embedding dim : {query_emb.shape[2]}")

In [ ]:
# Manual MaxSim computation
def maxsim_score(query_emb, page_emb):
    """Compute the MaxSim late-interaction score.

    query_emb : (num_query_tokens, dim)
    page_emb  : (num_patches, dim)
    Returns   : scalar score
    """
    sim = torch.matmul(
        query_emb.float(), page_emb.float().T
    )  # (num_query_tokens, num_patches)
    max_per_token = sim.max(dim=1).values  # (num_query_tokens,)
    return max_per_token.sum().item()

q = query_emb[0]  # (num_tokens, dim)

print(f"Query: '{query_text}'\n")
for idx, page_emb in enumerate(image_embeddings):
    score = maxsim_score(q, page_emb)
    print(f"  Page {idx} ({page_descriptions[idx]:>45s}) — MaxSim = {score:.2f}")

In [ ]:
# --- Use the built-in scoring utility for verification ---
scores = colpali_processor.score_multi_vector(query_emb, image_embeddings)
print("Scores from colpali_processor.score_multi_vector:")
for idx, s in enumerate(scores[0]):
    print(f"  Page {idx}: {s:.2f}")

In [ ]:
# --- Similarity heatmap: which patches match which query tokens ---
best_page_idx = int(scores[0].argmax())
page_emb = image_embeddings[best_page_idx]

# Compute full similarity matrix
sim_matrix = torch.matmul(
    q.float(), page_emb.float().T
).cpu().numpy()  # (num_query_tokens, num_patches)

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(sim_matrix, aspect="auto", cmap="viridis")
ax.set_xlabel("Page patch index")
ax.set_ylabel("Query token index")
ax.set_title(f"Similarity heatmap — query vs Page {best_page_idx} patches")
plt.colorbar(im, ax=ax, label="Cosine similarity")
plt.tight_layout()
plt.show()

print(f"\nBrighter spots indicate where a query token found its best-matching patch.")
print(f"This fine-grained alignment is what makes late interaction powerful.")

## Section 5 — Building a Retrieval Index with FAISS

In a production system you would not compute MaxSim over every page at query
time. Instead, you precompute patch embeddings and store them in a vector index.

Multi-vector representations make indexing trickier than single-vector models
because each page contributes **many** vectors. A common approach: store all
patch vectors in a flat FAISS index, maintain a mapping from patch indices to
page IDs, retrieve candidate patches at query time, group by page, and compute
exact MaxSim on the candidates.

In [ ]:
import faiss

# Flatten all page patches into a single matrix
all_patches = []
patch_to_page = []  # maps each row to its page index

for page_idx, emb in enumerate(image_embeddings):
    emb_np = emb.float().cpu().numpy()
    all_patches.append(emb_np)
    patch_to_page.extend([page_idx] * emb_np.shape[0])

all_patches = np.vstack(all_patches).astype("float32")
patch_to_page = np.array(patch_to_page)

# Normalise for cosine similarity
faiss.normalize_L2(all_patches)

dim = all_patches.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product = cosine on normalised vectors
index.add(all_patches)

print(f"FAISS index built — {index.ntotal} vectors, dim={dim}")

In [ ]:
def search_colpali(query_text, top_k=2):
    """Search the FAISS index and aggregate MaxSim-style scores per page."""
    batch_q = colpali_processor.process_queries([query_text]).to(colpali.device)
    with torch.no_grad():
        q_emb = colpali(**batch_q)[0].float().cpu().numpy()  # (tokens, dim)

    faiss.normalize_L2(q_emb)

    # For each query token, retrieve top patches
    n_probe = min(50, index.ntotal)
    distances, indices = index.search(q_emb, n_probe)  # (tokens, n_probe)

    # Aggregate: for each page, take the max similarity per query token, then sum
    page_scores = {}
    for tok_idx in range(q_emb.shape[0]):
        page_max = {}
        for rank in range(n_probe):
            pid = int(patch_to_page[indices[tok_idx, rank]])
            sim = float(distances[tok_idx, rank])
            if pid not in page_max or sim > page_max[pid]:
                page_max[pid] = sim
        for pid, sim in page_max.items():
            page_scores[pid] = page_scores.get(pid, 0.0) + sim

    ranked = sorted(page_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return ranked

print("search_colpali() defined.")

In [ ]:
# --- Run queries against the index ---
test_queries = [
    "What is the revenue for Q3 2024?",
    "How does the microservices architecture work?",
    "What is the parental leave policy?",
    "Compare attention mechanism complexity",
]

for query in test_queries:
    results = search_colpali(query, top_k=2)
    top_page = results[0][0]
    print(f"Query: '{query}'")
    for page_id, score in results:
        print(f"  → Page {page_id} ({page_descriptions[page_id]}) — score {score:.2f}")
    print()

In [ ]:
# --- Display the top result for a selected query ---
selected_query = "What is the revenue for Q3 2024?"
results = search_colpali(selected_query, top_k=1)
top_page_idx = results[0][0]
top_score = results[0][1]

fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(pages[top_page_idx])
ax.set_title(f"Top result for: '{selected_query}'\nPage {top_page_idx} — score {top_score:.2f}",
             fontsize=10)
ax.axis("off")
plt.tight_layout()
plt.show()

## Section 6 — ColPali vs Text-Based Retrieval

Let us build a text-embedding baseline using `all-MiniLM-L6-v2`. We embed
the **text descriptions** of each page (simulating OCR output) and compare
retrieval quality on queries where visual layout matters versus queries
where plain text suffices.

Text embeddings are tiny, fast, and excellent for narrative documents,
but cannot capture layout or spatial relationships. ColPali preserves
all of that at the cost of larger embeddings and more compute.

In [ ]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Simulated OCR text for each page (what a text pipeline would see)
ocr_texts = [
    "Acme Corp Q3 2024 Financial Report. Revenue grew 12% YoY driven by cloud. "
    "Operating margin 18.3%. Revenue 382M, Net Income 70M, EPS $1.40.",

    "System Architecture Overview. Microservices on Kubernetes, gRPC, Protocol Buffers, "
    "API gateway, Apache Kafka, PostgreSQL per service, Prometheus and Grafana.",

    "Employee Benefits Leave Policy. 20 days PTO, 10 sick days, parental leave 16 weeks "
    "primary 6 weeks secondary, $5000 tuition reimbursement, health dental vision.",

    "Efficient Attention Mechanisms Survey. Transformer self-attention quadratic cost. "
    "Linear attention, sparse patterns, low-rank. Linformer O(n) 97.3%, Performer 96.1%.",
]

text_embeddings = text_model.encode(ocr_texts, normalize_embeddings=True)
print(f"Text embeddings shape: {text_embeddings.shape}")

In [ ]:
def search_text(query, top_k=2):
    """Search using text embeddings with cosine similarity."""
    q_emb = text_model.encode([query], normalize_embeddings=True)
    sims = (q_emb @ text_embeddings.T)[0]
    ranked = np.argsort(-sims)[:top_k]
    return [(int(idx), float(sims[idx])) for idx in ranked]

# --- Side-by-side comparison ---
comparison_queries = [
    "What is the Q3 revenue figure?",
    "Explain the system architecture",
    "How many days of parental leave?",
    "Which attention method is most accurate?",
]

print(f"{'Query':<45} {'ColPali Top':<30} {'Text Top':<30}")
print("=" * 105)
for q in comparison_queries:
    cp_results = search_colpali(q, top_k=1)
    tx_results = search_text(q, top_k=1)
    cp_str = f"Page {cp_results[0][0]} ({cp_results[0][1]:.2f})"
    tx_str = f"Page {tx_results[0][0]} ({tx_results[0][1]:.4f})"
    print(f"{q:<45} {cp_str:<30} {tx_str:<30}")

In [ ]:
# --- Embedding size comparison ---
colpali_bytes = sum(e.float().cpu().numpy().nbytes for e in image_embeddings)
text_bytes = text_embeddings.nbytes

print("Storage comparison for 4 pages:\n")
print(f"  ColPali (multi-vector) : {colpali_bytes / 1024:.1f} KB  "
      f"({image_embeddings[0].shape[0]} patches × {image_embeddings[0].shape[1]}d per page)")
print(f"  Text (single-vector)   : {text_bytes / 1024:.1f} KB  "
      f"(1 vector × {text_embeddings.shape[1]}d per page)")
print(f"  Ratio                  : {colpali_bytes / text_bytes:.0f}×")

## Section 7 — Limitations and Practical Considerations

ColPali is a significant step forward, but it is not a universal replacement
for text-based retrieval. Engineers should weigh the following trade-offs:

| Consideration | ColPali (vision) | Text Embedding |
|---|---|---|
| **Compute per page** | ~200 ms on T4 GPU | ~5 ms on CPU |
| **Storage per page** | ~200 KB (multi-vector) | ~1.5 KB (single vector) |
| **Captures layout** | ✅ Full spatial info | ❌ Lost after OCR |
| **Captures tables** | ✅ Cell-level matching | ⚠️ Depends on OCR quality |
| **Text-heavy docs** | Good | Excellent |
| **GPU required** | Yes (inference) | No |

### When to Use Each Approach

* **Use ColPali** when documents are visually rich: financial reports with tables,
  research papers with charts, slide decks, scanned forms with logos and stamps.
* **Use text embeddings** when documents are narrative prose with little layout
  complexity, or when you need to run on CPU at very low cost.
* **Use both** — the hybrid approach in the next notebook
  (`11_hybrid_text_plus_vision_retrieval.ipynb`) combines visual and text signals.

For ColPali in a full RAG pipeline, see also
`rag/multi_model_rag_with_colpali.ipynb`, which chains ColPali retrieval with a
generative model for answer synthesis.

## 🏋️ Exercises

**Exercise 1 — Add a chart page.**  
Create a synthetic page that contains a bar-chart image (drawn with Pillow or
matplotlib). Add it to the corpus, re-index, and test a query like
*"Which product had the highest sales?"*. Does ColPali retrieve the chart page?

**Exercise 2 — Patch-level attribution.**  
For a given query, identify the top-3 patches (by MaxSim contribution) on the
retrieved page. Overlay bounding boxes on the page image to visualise which
regions the model relied on.

**Exercise 3 — Scaling experiment.**  
Generate 50 synthetic pages with varied content. Measure indexing time, search
latency, and index size. Plot how each metric scales with corpus size.

In [ ]:
# --- Exercise 1 starter code ---
# Create a page with a simple bar chart

def create_chart_page(title, categories, values, width=800, height=1100):
    """Create a document page containing a bar chart."""
    fig_chart, ax_chart = plt.subplots(figsize=(6, 3))
    ax_chart.bar(categories, values, color=["#4285f4", "#ea4335", "#fbbc05", "#34a853"])
    ax_chart.set_title(title)
    ax_chart.set_ylabel("Sales ($M)")
    fig_chart.tight_layout()
    fig_chart.savefig("_temp_chart.png", dpi=100)
    plt.close(fig_chart)

    chart_img = Image.open("_temp_chart.png")
    page = Image.new("RGB", (width, height), "white")
    page.paste(chart_img, (40, 60))
    os.remove("_temp_chart.png")
    return page

# TODO: Create the chart page, encode it, add to the index, and search
# chart_page = create_chart_page("Product Sales 2024",
#                                ["Widget A", "Widget B", "Widget C", "Widget D"],
#                                [120, 95, 180, 60])
print("Exercise 1: extend the code above to complete the exercise.")

In [ ]:
# --- Exercise 2 starter code ---
# Identify top patches by MaxSim contribution

def get_top_patches(query_emb, page_emb, top_n=3):
    """Return the top-N patch indices that contribute most to MaxSim."""
    sim = torch.matmul(query_emb.float(), page_emb.float().T)  # (tokens, patches)
    max_vals, max_idxs = sim.max(dim=1)  # best patch per query token
    # Sort query tokens by their max similarity (highest contribution first)
    sorted_tok = torch.argsort(-max_vals)[:top_n]
    return [(int(max_idxs[t]), float(max_vals[t])) for t in sorted_tok]

# TODO: Use get_top_patches with a query and page, then draw bounding boxes
print("Exercise 2: use get_top_patches() and overlay bounding boxes on the page image.")

## 📝 Key Takeaways

| # | Insight |
|---|---|
| 1 | ColPali embeds the **raw page image**, skipping OCR entirely and preserving layout, tables, and figures. |
| 2 | Each page produces a **multi-vector** representation — one embedding per spatial patch — enabling fine-grained matching. |
| 3 | **MaxSim** (from ColBERT) aligns each query token with its best-matching page patch, then sums the scores. |
| 4 | Late interaction keeps query and document encoding **independent**, allowing precomputation and caching of page embeddings. |
| 5 | ColPali shines on **visually rich documents** (tables, charts, forms) where OCR loses critical information. |
| 6 | The trade-off is **larger storage** (~100× vs single-vector text) and **GPU compute** — hybrid systems combine the best of both approaches. |

## 📚 References

1. **ColPali** — Faysse, M., Music, H., Music, A., Music, J. (2024). *ColPali: Efficient Document Retrieval with Vision Language Models.* [arXiv:2407.01272](https://arxiv.org/abs/2407.01272)
2. **ColBERT** — Khattab, O., & Zaharia, M. (2020). *ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction over BERT.* [arXiv:2004.12832](https://arxiv.org/abs/2004.12832)
3. **PaliGemma** — Beyer, L., et al. (2024). *PaliGemma: A versatile 3B VLM for transfer.* [arXiv:2407.07726](https://arxiv.org/abs/2407.07726)
4. **FAISS** — Johnson, J., Douze, M., & Jégou, H. (2019). *Billion-scale similarity search with GPUs.* IEEE Transactions on Big Data.
5. **Sentence-Transformers** — Reimers, N., & Gurevych, I. (2019). *Sentence-BERT.* [arXiv:1908.10084](https://arxiv.org/abs/1908.10084)

---

⬅️ [09 — Captioning as a Text Bridge](09_captioning_as_a_text_bridge.ipynb) · 
[11 — Hybrid Text + Vision Retrieval](11_hybrid_text_plus_vision_retrieval.ipynb) ➡️